In [2]:
# 1. Instalasi dan Persiapan PySpark
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Memulai Spark Session...")
spark = SparkSession.builder \
    .appName("Farmasi_Priyanto_15240017") \
    .getOrCreate()

# Memuat dataset CSV
df_spark = spark.read.csv('Dataset_Farmasi_Priyanto_15240017.csv', header=True, inferSchema=True)

# 2. Quality Checks (Membuang data kosong dan mengubah format tanggal)
print("\nMelakukan Quality Checks...")
df_clean = df_spark.dropna(subset=["transaction_id", "product_id"]) \
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd")) \
    .withColumn("expiry_date", F.to_date(F.col("expiry_date"), "yyyy-MM-dd"))

# 3. Transformasi Kolom Turunan (Menghitung risiko kedaluwarsa dan total nilai)
print("\nMembuat Kolom Turunan (Transformasi Bisnis)...")
df_transform = df_clean \
    .withColumn("days_to_expire", F.datediff(F.col("expiry_date"), F.col("transaction_date"))) \
    .withColumn("total_value", F.col("quantity") * F.col("unit_price")) \
    .withColumn(
        "expiry_risk_status",
        F.when(F.col("days_to_expire") < 90, "Risiko Tinggi (Exp < 3 Bulan)")
         .when(F.col("days_to_expire") < 180, "Risiko Sedang (Exp < 6 Bulan)")
         .otherwise("Aman")
    )

print("--- Contoh Hasil Transformasi ---")
df_transform.select("transaction_id", "product_id", "days_to_expire", "expiry_risk_status", "total_value").show(5)

# 4. Agregasi Data (Merekap jumlah penjualan per Gudang dan Kategori)
print("\nMelakukan Agregasi Data...")
df_agg = df_transform.filter(F.col("transaction_type") == "Penjualan") \
    .groupBy("warehouse", "product_category") \
    .agg(
        F.sum("quantity").alias("total_qty_terjual"),
        F.sum("total_value").alias("total_pendapatan")
    ) \
    .orderBy(F.desc("total_pendapatan"))

df_agg.show(5)

# 5. Menyimpan ke Format Parquet (Wajib untuk Notebook ke-3 nanti)
output_path = "output_farmasi_parquet"
df_transform.write.mode("overwrite").parquet(output_path)
print(f"\nSelesai! Data berhasil dipadatkan dan disimpan di folder: {output_path}")

Memulai Spark Session...

Melakukan Quality Checks...

Membuat Kolom Turunan (Transformasi Bisnis)...
--- Contoh Hasil Transformasi ---
+--------------+--------------------+--------------+------------------+-----------+
|transaction_id|          product_id|days_to_expire|expiry_risk_status|total_value|
+--------------+--------------------+--------------+------------------+-----------+
|   TRX-0000001|    Obat_Analgesik_6|           573|              Aman|    6435308|
|   TRX-0000002|    Obat_Analgesik_2|           427|              Aman|    1691340|
|   TRX-0000003|Obat_Sirup Obat B...|           694|              Aman|    6473456|
|   TRX-0000004|   Obat_Antibiotik_7|           210|              Aman|    1661151|
|   TRX-0000005|   Obat_Antibiotik_6|           351|              Aman|    3686408|
+--------------+--------------------+--------------+------------------+-----------+
only showing top 5 rows

Melakukan Agregasi Data...
+--------------+------------------+-----------------+---

In [3]:
import shutil
from google.colab import files

# Nama folder Parquet yang sudah dibuat oleh PySpark
output_path = "output_farmasi_parquet"

print("Membungkus folder Parquet menjadi file ZIP...")
shutil.make_archive(output_path, 'zip', output_path)

print("Memulai proses unduhan ke komputer...")
files.download(output_path + '.zip')

Membungkus folder Parquet menjadi file ZIP...
Memulai proses unduhan ke komputer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>